# NanoVLM for MiniGrid-EmptyEnv: Reproducible Training Notebook

This notebook documents a staged training pipeline for a multimodal policy based on NanoVLM in the MiniGrid-Empty-6x6 environment. The workflow is designed for reproducible execution in Google Colab with GPU support and includes artifact persistence for fault-tolerant resumption.

## 1) Environment Initialization and Execution Preconditions

Run cells strictly from top to bottom.

- In Colab: enable a GPU runtime before starting.
- In local VS Code: activate the target Python environment and verify dependencies.

This section establishes a consistent execution context (repository root, interpreter, and hardware availability) required for valid experimental comparisons.

In [2]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/SergeySolovyev/T-Lab-2026.-Multimodal-VLMs.git'
REPO_DIR = Path('/content/T-Lab-2026.-Multimodal-VLMs')

def find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'requirements.txt').exists() and (p / 'src').exists():
            return p
    for p in [REPO_DIR, Path('/content/T-bank'), Path('/content/repo')]:
        if (p / 'requirements.txt').exists() and (p / 'src').exists():
            return p
    if Path('/content').exists():
        for req in Path('/content').rglob('requirements.txt'):
            if (req.parent / 'src').exists():
                return req.parent
    return Path.cwd()

project_root = find_project_root()

# In Colab: clone if needed, always pull latest
if Path('/content').exists():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
    project_root = REPO_DIR

os.chdir(project_root)
print('Project root:', project_root)
print('Python:', sys.version)
print('Executable:', sys.executable)

Project root: /content/T-Lab-2026.-Multimodal-VLMs
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Executable: /usr/bin/python3


: 

In [5]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)
else:
    print('GPU не найден. Для обучения нужен Colab GPU runtime.')

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
CUDA version: 12.8


In [3]:
from pathlib import Path
import shutil

IS_COLAB = Path('/content').exists()
DRIVE_MOUNT_ROOT = Path('/content/drive')
DRIVE_PROJECT_NAME = 'T-Lab-2026.-Multimodal-VLMs'
DRIVE_ARTIFACTS_DIR = None

if IS_COLAB:
    try:
        from google.colab import drive  # type: ignore
        drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=False)

        mydrive = DRIVE_MOUNT_ROOT / 'MyDrive'
        DRIVE_ARTIFACTS_DIR = mydrive / DRIVE_PROJECT_NAME / 'artifacts'
        DRIVE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

        local_artifacts = Path('artifacts')
        if DRIVE_ARTIFACTS_DIR.exists() and any(DRIVE_ARTIFACTS_DIR.iterdir()):
            shutil.copytree(DRIVE_ARTIFACTS_DIR, local_artifacts, dirs_exist_ok=True)
            print(f'Restored artifacts from Drive: {DRIVE_ARTIFACTS_DIR}')
        else:
            local_artifacts.mkdir(parents=True, exist_ok=True)
            print(f'Drive artifacts folder is empty (first run): {DRIVE_ARTIFACTS_DIR}')

    except Exception as exc:
        print('Google Drive mount skipped or failed:', exc)
else:
    print('Not in Colab runtime: Drive mount skipped.')


def backup_artifacts_to_drive() -> None:
    local_artifacts = Path('artifacts')
    if not local_artifacts.exists():
        print('No local artifacts to backup yet.')
        return
    if DRIVE_ARTIFACTS_DIR is None:
        print('Drive is not mounted, backup skipped.')
        return
    shutil.copytree(local_artifacts, DRIVE_ARTIFACTS_DIR, dirs_exist_ok=True)
    print(f'Artifacts backed up to Drive: {DRIVE_ARTIFACTS_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive artifacts folder is empty (first run): /content/drive/MyDrive/T-Lab-2026.-Multimodal-VLMs/artifacts


## 2) Artifact Persistence and Recovery Strategy

To mitigate runtime interruptions (common in hosted GPU sessions), artifacts are synchronized with Google Drive and restored on restart.

The corresponding code cell mounts Drive, restores local `artifacts/` when available, and defines a backup helper for incremental checkpoints and metrics. This enables continuation of long experiments without re-running completed stages.

In [7]:
import subprocess
import sys
from pathlib import Path

req = Path('requirements.txt')
if not req.exists():
    raise FileNotFoundError(
        f'requirements.txt не найден. Текущая папка: {Path.cwd()}\n'
        'Сначала выполните предыдущую ячейку определения project_root.'
    )

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req)], check=True)
print('Dependencies installed ✅')

Dependencies installed ✅


## 3) Model Preparation and Artifact Directory Layout

This stage prepares the NanoVLM codebase (`external/nanoVLM`), applies project-specific policy integration, and creates a standardized artifact structure.

A fixed directory schema is used to ensure traceability of datasets, checkpoints, and plots across SFT and GRPO phases.

In [8]:
import subprocess, json
from pathlib import Path

# 1) Clone/update nanoVLM
nanovlm_dir = Path('external/nanoVLM')
nanovlm_dir.parent.mkdir(parents=True, exist_ok=True)
if not nanovlm_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/huggingface/nanoVLM.git', str(nanovlm_dir)], check=True)
    print('Cloned nanoVLM →', nanovlm_dir)
else:
    subprocess.run(['git', '-C', str(nanovlm_dir), 'pull'], check=False)
    print('nanoVLM already exists →', nanovlm_dir)

# 2) Patch nanoVLM config loader (ignore unknown keys)
vlm_file = nanovlm_dir / 'models' / 'vision_language_model.py'
text = vlm_file.read_text(encoding='utf-8')
old_line = 'cfg = VLMConfig(**json.load(f))'
if old_line in text:
    new_lines = (
        'cfg_dict = json.load(f)\n'
        '        valid_keys = set(VLMConfig.__dataclass_fields__.keys())\n'
        '        cfg = VLMConfig(**{k: v for k, v in cfg_dict.items() if k in valid_keys})'
    )
    text = text.replace(old_line, new_lines)

# 3) Patch image-token replacement to avoid shape mismatch crashes
old_replace = (
    '        mask = (input_ids == self.tokenizer.image_token_id)\n'
    '        updated_token_embd[mask] = image_embd.view(-1, image_embd.size(-1)).to(updated_token_embd.dtype) # torch flattens before assigning\n'
)
new_replace = (
    '        mask = (input_ids == self.tokenizer.image_token_id)\n'
    '        flat = image_embd.view(-1, image_embd.size(-1)).to(updated_token_embd.dtype)\n'
    '        slot_idx = mask.nonzero(as_tuple=False)\n'
    '        n = min(slot_idx.size(0), flat.size(0))\n'
    '        if n > 0:\n'
    '            updated_token_embd[slot_idx[:n, 0], slot_idx[:n, 1]] = flat[:n]\n'
)
if old_replace in text:
    text = text.replace(old_replace, new_replace)

vlm_file.write_text(text, encoding='utf-8')
print('Patched vision_language_model.py ✅')

# 4) Overwrite nanovlm_policy with CUDA-stable single-patch implementation
policy_file = Path('src/models/nanovlm_policy.py')
policy_code = '''from __future__ import annotations

import re
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from PIL import Image

from src.nano_imports import add_nanovlm_to_path


class NanoVLMPolicy:
    def __init__(
        self,
        model_source: str,
        nanovlm_repo: str,
        device: str = "cuda",
    ) -> None:
        add_nanovlm_to_path(nanovlm_repo)

        from models.vision_language_model import VisionLanguageModel
        from data.processors import get_image_processor, get_image_string, get_tokenizer

        self._get_image_string = get_image_string
        self.model = VisionLanguageModel.from_pretrained(model_source)
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.train()

        # SINGLE-PATCH MODE: MiniGrid frames are tiny (~96px),
        # multi-patch splitting is pointless and causes tokenizer/vocab crashes.
        patch = self.model.cfg.vit_img_size          # 224
        self.model.cfg.max_img_size = patch           # force exactly 1 patch

        self.tokenizer = get_tokenizer(
            self.model.cfg.lm_tokenizer,
            self.model.cfg.vlm_extra_tokens,
            self.model.cfg.lm_chat_template,
        )

        self._ensure_vocab_alignment()

        resize_to_max_side_len = getattr(self.model.cfg, "resize_to_max_side_len", False)
        self.image_processor = get_image_processor(
            self.model.cfg.max_img_size,
            self.model.cfg.vit_img_size,
            resize_to_max_side_len,
        )

    def _ensure_vocab_alignment(self) -> None:
        embed = self.model.decoder.token_embedding
        model_vocab = embed.num_embeddings
        tok_vocab = len(self.tokenizer)
        if tok_vocab <= model_vocab:
            return

        hidden = embed.embedding_dim
        device = embed.weight.device
        dtype = embed.weight.dtype

        new_embed = torch.nn.Embedding(tok_vocab, hidden, device=device, dtype=dtype)
        with torch.no_grad():
            new_embed.weight[:model_vocab].copy_(embed.weight)
            filler = embed.weight.mean(dim=0, keepdim=True)
            new_embed.weight[model_vocab:].copy_(filler)
        self.model.decoder.token_embedding = new_embed

        if hasattr(self.model.decoder, "head"):
            old_head = self.model.decoder.head
            new_head = torch.nn.Linear(hidden, tok_vocab, bias=False, device=device, dtype=dtype)
            with torch.no_grad():
                rows = min(old_head.weight.shape[0], model_vocab)
                new_head.weight[:rows].copy_(old_head.weight[:rows])
                if tok_vocab > rows:
                    new_head.weight[rows:].copy_(new_embed.weight[rows:])
            self.model.decoder.head = new_head

        if getattr(self.model.cfg, "lm_tie_weights", False) and hasattr(self.model.decoder, "head"):
            self.model.decoder.head.weight = self.model.decoder.token_embedding.weight

        self.model.cfg.lm_vocab_size = tok_vocab

    def set_mode(self, train: bool) -> None:
        self.model.train(train)

    def save(self, path: str) -> None:
        Path(path).mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(path)

    @classmethod
    def load_from_checkpoint(cls, checkpoint_path: str, nanovlm_repo: str, device: str = "cuda") -> "NanoVLMPolicy":
        return cls(model_source=checkpoint_path, nanovlm_repo=nanovlm_repo, device=device)

    def _encode_sample(self, frame: np.ndarray, user_prompt: str, assistant_text: str):
        image = Image.fromarray(frame)
        processed_images, split_ratio = self.image_processor(image)
        if (
            not hasattr(self.tokenizer, "global_image_token")
            and split_ratio[0] * split_ratio[1] == len(processed_images) - 1
        ):
            processed_images = processed_images[1:]

        image_string = self._get_image_string(self.tokenizer, [split_ratio], self.model.cfg.mp_image_token_length)

        user_messages = [{"role": "user", "content": image_string + user_prompt}]
        full_messages = [
            {"role": "user", "content": image_string + user_prompt},
            {"role": "assistant", "content": assistant_text},
        ]

        prompt_text = self.tokenizer.apply_chat_template(user_messages, tokenize=False, add_generation_prompt=True)
        full_text = self.tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)

        prompt_ids = self.tokenizer(prompt_text, return_tensors="pt")
        full_ids = self.tokenizer(full_text, return_tensors="pt")

        input_ids = full_ids["input_ids"]
        attention_mask = full_ids["attention_mask"]
        labels = input_ids.clone()
        labels[:, : prompt_ids["input_ids"].shape[1]] = -100

        # Safety: clamp token ids to valid range
        vocab_size = self.model.decoder.token_embedding.num_embeddings
        input_ids = input_ids.clamp(0, vocab_size - 1)

        return {
            "input_ids": input_ids.to(self.device),
            "attention_mask": attention_mask.to(self.device),
            "labels": labels.to(self.device),
            "images": processed_images.to(self.device),
        }

    def sft_loss(self, frame: np.ndarray, user_prompt: str, assistant_text: str) -> torch.Tensor:
        batch = self._encode_sample(frame, user_prompt, assistant_text)
        _logits, loss = self.model(
            batch["input_ids"],
            batch["images"],
            attention_mask=batch["attention_mask"],
            targets=batch["labels"],
        )
        return loss

    @torch.no_grad()
    def generate(self, frame: np.ndarray, user_prompt: str, max_new_tokens: int = 64, temperature: float = 0.0) -> str:
        self.model.eval()
        image = Image.fromarray(frame)
        processed_images, split_ratio = self.image_processor(image)
        if (
            not hasattr(self.tokenizer, "global_image_token")
            and split_ratio[0] * split_ratio[1] == len(processed_images) - 1
        ):
            processed_images = processed_images[1:]

        image_string = self._get_image_string(self.tokenizer, [split_ratio], self.model.cfg.mp_image_token_length)
        messages = [{"role": "user", "content": image_string + user_prompt}]
        prompt_text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        encoded = self.tokenizer(prompt_text, return_tensors="pt")

        # Safety: clamp token ids
        vocab_size = self.model.decoder.token_embedding.num_embeddings
        input_ids = encoded["input_ids"].clamp(0, vocab_size - 1).to(self.device)

        generated_ids = self.model.generate(
            input_ids=input_ids,
            images=processed_images.to(self.device),
            attention_mask=encoded["attention_mask"].to(self.device),
            max_new_tokens=max_new_tokens,
            greedy=temperature <= 1e-8,
            temperature=max(temperature, 1e-6),
        )
        text = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        self.model.train()
        return text.strip()


def parse_action(text: str) -> Optional[str]:
    action_line = re.search(r"ACTION\\s*:\\s*(left|right|forward)", text, re.IGNORECASE)
    if action_line:
        return action_line.group(1).lower()
    any_action = re.search(r"\\b(left|right|forward)\\b", text, re.IGNORECASE)
    if any_action:
        return any_action.group(1).lower()
    return None
'''
policy_file.write_text(policy_code, encoding='utf-8')
print('Overwritten nanovlm_policy.py with single-patch version ✅')

# 5) Artifact dirs
for d in ['artifacts/datasets', 'artifacts/sft', 'artifacts/grpo_action', 'artifacts/grpo_text_action', 'artifacts/plots']:
    Path(d).mkdir(parents=True, exist_ok=True)

print('Setup done ✅')

Cloned nanoVLM → external/nanoVLM
Patched vision_language_model.py ✅
Overwritten nanovlm_policy.py with single-patch version ✅
Setup done ✅


## 4) Staged Training Protocol (SFT → GRPO)

The pipeline is intentionally decomposed into independent stages:

- 4.0 — global configuration
- 4.1 — expert dataset generation
- 4.2 — supervised fine-tuning (SFT)
- 4.3 — GRPO in `action` mode
- 4.4 — GRPO in `text_action` mode
- 4.5 — cross-run comparison

This design improves robustness and reproducibility: after an interruption, only missing stages are re-executed, while completed artifacts remain valid.

In [4]:
import os, sys, subprocess

RUN_PROFILE = 'mvp'   # 'mvp' | 'full'
MODEL_SOURCE = 'lusxvr/nanoVLM-222M'

cfg = dict(
    mvp=dict(train=1200, val=200, test=200, epochs=1, updates=40, epu=4, gs=4, eval_ep=20),
    full=dict(train=20000, val=2000, test=2000, epochs=5, updates=300, epu=8, gs=8, eval_ep=100),
)[RUN_PROFILE]

py = sys.executable
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

def run_cmd(cmd):
    print(f'\n{"="*60}')
    print(f'>>> {" ".join(cmd)}')
    print(f'{"="*60}', flush=True)
    result = subprocess.run(cmd, env=env)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed (rc={result.returncode})')

print('Config ready ✅')
print('Run next cells 4.1 → 4.5 one by one')

Config ready ✅
Run next cells 4.1 → 4.5 one by one


In [10]:
# 4.1 Generate expert dataset
run_cmd([
    py, '-m', 'src.data.generate_expert_dataset',
    '--out_dir', 'artifacts/datasets',
    '--train_episodes', str(cfg['train']),
    '--val_episodes',   str(cfg['val']),
    '--test_episodes',  str(cfg['test']),
    '--seed', '42', '--map_sizes', '6',
])

print('Dataset ready ✅')


>>> /usr/bin/python3 -m src.data.generate_expert_dataset --out_dir artifacts/datasets --train_episodes 1200 --val_episodes 200 --test_episodes 200 --seed 42 --map_sizes 6
Dataset ready ✅


In [11]:
# 4.2 SFT
run_cmd([
    py, '-m', 'src.train.train_sft',
    '--train_npz', 'artifacts/datasets/train.npz',
    '--val_npz',   'artifacts/datasets/val.npz',
    '--mode', 'action', '--epochs', str(cfg['epochs']),
    '--eval_episodes', str(cfg['eval_ep']),
    '--output_dir', 'artifacts/sft',
    '--model_source', MODEL_SOURCE,
    '--nanovlm_repo', 'external/nanoVLM',
])

print('SFT finished ✅')


>>> /usr/bin/python3 -m src.train.train_sft --train_npz artifacts/datasets/train.npz --val_npz artifacts/datasets/val.npz --mode action --epochs 1 --eval_episodes 20 --output_dir artifacts/sft --model_source lusxvr/nanoVLM-222M --nanovlm_repo external/nanoVLM
SFT finished ✅


In [6]:
# 4.2.1 Backup artifacts after SFT
try:
    backup_artifacts_to_drive()
except NameError:
    print('backup_artifacts_to_drive is not defined; run the Drive mount cell first.')

Artifacts backed up to Drive: /content/drive/MyDrive/T-Lab-2026.-Multimodal-VLMs/artifacts


In [12]:
# 4.3 GRPO (action)
run_cmd([
    py, '-m', 'src.train.train_grpo',
    '--init_checkpoint', 'artifacts/sft/checkpoint_last',
    '--mode', 'action',
    '--updates', str(cfg['updates']),
    '--episodes_per_update', str(cfg['epu']),
    '--group_size', str(cfg['gs']),
    '--eval_episodes', str(cfg['eval_ep']),
    '--output_dir', 'artifacts/grpo_action',
    '--nanovlm_repo', 'external/nanoVLM',
])

print('GRPO action finished ✅')


>>> /usr/bin/python3 -m src.train.train_grpo --init_checkpoint artifacts/sft/checkpoint_last --mode action --updates 40 --episodes_per_update 4 --group_size 4 --eval_episodes 20 --output_dir artifacts/grpo_action --nanovlm_repo external/nanoVLM


: 

In [ ]:
# 4.4 GRPO (text_action)
run_cmd([
    py, '-m', 'src.train.train_grpo',
    '--init_checkpoint', 'artifacts/sft/checkpoint_last',
    '--mode', 'text_action',
    '--updates', str(cfg['updates']),
    '--episodes_per_update', str(cfg['epu']),
    '--group_size', str(cfg['gs']),
    '--eval_episodes', str(cfg['eval_ep']),
    '--output_dir', 'artifacts/grpo_text_action',
    '--nanovlm_repo', 'external/nanoVLM',
])

print('GRPO text_action finished ✅')


>>> /usr/bin/python3 -m src.train.train_grpo --init_checkpoint artifacts/sft/checkpoint_last --mode text_action --updates 40 --episodes_per_update 4 --group_size 4 --eval_episodes 20 --output_dir artifacts/grpo_text_action --nanovlm_repo external/nanoVLM


In [5]:
from pathlib import Path
import pandas as pd

required = [
    'artifacts/sft/history.csv',
    'artifacts/grpo_action/history.csv',
    'artifacts/grpo_text_action/history.csv',
    'artifacts/plots/summary_table.csv',
    'artifacts/plots/sample_efficiency.csv',
    'artifacts/plots/success_rate.png',
    'artifacts/plots/return.png',
]

for path in required:
    p = Path(path)
    print(f'{path}:', 'OK' if p.exists() else 'MISSING')

summary_path = Path('artifacts/plots/summary_table.csv')
if summary_path.exists():
    print('\nSummary table:')
    display(pd.read_csv(summary_path))

sample_path = Path('artifacts/plots/sample_efficiency.csv')
if sample_path.exists():
    print('\nSample efficiency:')
    display(pd.read_csv(sample_path))

artifacts/sft/history.csv: OK
artifacts/grpo_action/history.csv: OK
artifacts/grpo_text_action/history.csv: MISSING
artifacts/plots/summary_table.csv: MISSING
artifacts/plots/sample_efficiency.csv: MISSING
artifacts/plots/success_rate.png: MISSING
artifacts/plots/return.png: MISSING


: 

In [ ]:
# 4.6 Resume: run ONLY missing stages (4.3 -> 4.4 -> 4.5)
from pathlib import Path

missing_runtime = [name for name in ('py', 'cfg', 'run_cmd') if name not in globals()]
if missing_runtime:
    raise RuntimeError(
        f"Missing runtime objects: {missing_runtime}. "
        "Run cells: project root, Drive mount, setup/config (up to 4.0), then retry."
    )

paths = {
    'sft_history': Path('artifacts/sft/history.csv'),
    'grpo_action_history': Path('artifacts/grpo_action/history.csv'),
    'grpo_text_history': Path('artifacts/grpo_text_action/history.csv'),
    'summary_table': Path('artifacts/plots/summary_table.csv'),
    'sample_efficiency': Path('artifacts/plots/sample_efficiency.csv'),
    'success_plot': Path('artifacts/plots/success_rate.png'),
    'return_plot': Path('artifacts/plots/return.png'),
}

if not paths['sft_history'].exists():
    raise FileNotFoundError(
        "Missing artifacts/sft/history.csv. Run 4.2 SFT first (or restore artifacts from Drive)."
    )

# 4.3
if not paths['grpo_action_history'].exists():
    print('Missing GRPO action history -> running 4.3 ...')
    run_cmd([
        py, '-m', 'src.train.train_grpo',
        '--init_checkpoint', 'artifacts/sft/checkpoint_last',
        '--mode', 'action',
        '--updates', str(cfg['updates']),
        '--episodes_per_update', str(cfg['epu']),
        '--group_size', str(cfg['gs']),
        '--eval_episodes', str(cfg['eval_ep']),
        '--output_dir', 'artifacts/grpo_action',
        '--nanovlm_repo', 'external/nanoVLM',
    ])
else:
    print('GRPO action history exists -> skip 4.3')

# 4.4
if not paths['grpo_text_history'].exists():
    print('Missing GRPO text_action history -> running 4.4 ...')
    run_cmd([
        py, '-m', 'src.train.train_grpo',
        '--init_checkpoint', 'artifacts/sft/checkpoint_last',
        '--mode', 'text_action',
        '--updates', str(cfg['updates']),
        '--episodes_per_update', str(cfg['epu']),
        '--group_size', str(cfg['gs']),
        '--eval_episodes', str(cfg['eval_ep']),
        '--output_dir', 'artifacts/grpo_text_action',
        '--nanovlm_repo', 'external/nanoVLM',
    ])
else:
    print('GRPO text_action history exists -> skip 4.4')

# 4.5
plot_targets = [
    paths['summary_table'],
    paths['sample_efficiency'],
    paths['success_plot'],
    paths['return_plot'],
]
if not all(p.exists() for p in plot_targets):
    print('Missing comparison outputs -> running 4.5 ...')
    run_cmd([
        py, '-m', 'src.eval.compare_runs',
        '--sft',              'artifacts/sft/history.csv',
        '--grpo_action',      'artifacts/grpo_action/history.csv',
        '--grpo_text_action', 'artifacts/grpo_text_action/history.csv',
        '--out_dir',          'artifacts/plots',
        '--success_threshold', '0.80',
        '--episodes_per_update', str(cfg['epu']),
        '--group_size', str(cfg['gs']),
        '--max_steps', '100',
    ])
else:
    print('Comparison outputs already exist -> skip 4.5')

print('\nFinal status:')
for _, path in paths.items():
    print(f'{path}:', 'OK' if path.exists() else 'MISSING')

if 'backup_artifacts_to_drive' in globals():
    backup_artifacts_to_drive()
else:
    print('backup_artifacts_to_drive() is not available in this runtime.')

: 

: 

## 5) Result Inspection and Integrity Checks

After training, verify that all required artifacts are present (histories, summary tables, and figures). This step is a minimal quality gate before drawing conclusions or transferring results into the report.

If any artifact is missing, resume execution from the corresponding stage instead of restarting the entire pipeline.

In [ ]:
from pathlib import Path
import pandas as pd

required = [
    'artifacts/sft/history.csv',
    'artifacts/grpo_action/history.csv',
    'artifacts/grpo_text_action/history.csv',
    'artifacts/plots/summary_table.csv',
    'artifacts/plots/success_rate.png',
]

for path in required:
    p = Path(path)
    print(f'{path}:', 'OK' if p.exists() else 'MISSING')

summary_path = Path('artifacts/plots/summary_table.csv')
if summary_path.exists():
    display(pd.read_csv(summary_path))

## 6) Version Control and Reproducibility Release

This section synchronizes the current experimental state with a remote Git repository.

Best-practice note: avoid committing large binary checkpoints directly to Git history; keep code and lightweight metadata in the repository, and store heavy artifacts separately (e.g., external storage or release assets).

In [ ]:
import subprocess

YOUR_REPO_URL = 'https://github.com/SergeySolovyev/T-Lab-2026.-Multimodal-VLMs.git'

commands = [
    ['git', 'status'],
    ['git', 'add', '.'],
    ['git', 'commit', '-m', 'MiniGrid EmptyEnv + NanoVLM SFT/GRPO pipeline'],
    ['git', 'branch', '-M', 'main'],
    ['git', 'remote', 'remove', 'origin'],
    ['git', 'remote', 'add', 'origin', YOUR_REPO_URL],
    ['git', 'push', '-u', 'origin', 'main'],
]

for cmd in commands:
    print('>>>', ' '.join(cmd))
    result = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

## 7) Multi-Seed Evaluation Protocol

Single-seed outcomes can be noisy in reinforcement learning. This section runs repeated experiments with different random seeds and stores outputs in `artifacts/multiseed/seed_<N>/`.

Recommended workflow:

- start with a small smoke test (e.g., 2 seeds),
- scale to 5+ seeds for statistically meaningful comparison.

In [ ]:
import subprocess, sys
from pathlib import Path

SEEDS = [11, 22]       # smoke-test на 2 seeds, потом увеличить до 5+
RUN_PROFILE = 'mvp'    # 'mvp' | 'full'
MODEL_SOURCE = 'lusxvr/nanoVLM-222M'
BASE_DIR = Path('artifacts/multiseed')

cfg = dict(
    mvp=dict(train=4000, val=500, test=500, epochs=2, updates=100, epu=4, gs=4),
    full=dict(train=20000, val=2000, test=2000, epochs=5, updates=300, epu=8, gs=8),
)[RUN_PROFILE]

py = sys.executable

for seed in SEEDS:
    sd = BASE_DIR / f'seed_{seed}'
    dirs = {k: sd / k for k in ['datasets', 'sft', 'grpo_action', 'grpo_text_action', 'plots']}
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)

    cmds = [
        [py, '-m', 'src.data.generate_expert_dataset',
         '--out_dir', str(dirs['datasets']),
         '--train_episodes', str(cfg['train']),
         '--val_episodes', str(cfg['val']),
         '--test_episodes', str(cfg['test']),
         '--seed', str(seed), '--map_sizes', '6'],

        [py, '-m', 'src.train.train_sft',
         '--train_npz', str(dirs['datasets'] / 'train.npz'),
         '--val_npz',   str(dirs['datasets'] / 'val.npz'),
         '--mode', 'action', '--epochs', str(cfg['epochs']),
         '--seed', str(seed),
         '--output_dir', str(dirs['sft']),
         '--model_source', MODEL_SOURCE,
         '--nanovlm_repo', 'external/nanoVLM'],

        [py, '-m', 'src.train.train_grpo',
         '--init_checkpoint', str(dirs['sft'] / 'checkpoint_last'),
         '--mode', 'action',
         '--updates', str(cfg['updates']),
         '--episodes_per_update', str(cfg['epu']),
         '--group_size', str(cfg['gs']),
         '--seed', str(seed),
         '--output_dir', str(dirs['grpo_action']),
         '--nanovlm_repo', 'external/nanoVLM'],

        [py, '-m', 'src.train.train_grpo',
         '--init_checkpoint', str(dirs['sft'] / 'checkpoint_last'),
         '--mode', 'text_action',
         '--updates', str(cfg['updates']),
         '--episodes_per_update', str(cfg['epu']),
         '--group_size', str(cfg['gs']),
         '--seed', str(seed),
         '--output_dir', str(dirs['grpo_text_action']),
         '--nanovlm_repo', 'external/nanoVLM'],

        [py, '-m', 'src.eval.compare_runs',
         '--sft',              str(dirs['sft'] / 'history.csv'),
         '--grpo_action',      str(dirs['grpo_action'] / 'history.csv'),
         '--grpo_text_action', str(dirs['grpo_text_action'] / 'history.csv'),
         '--out_dir',          str(dirs['plots'])],
    ]

    print(f'\n===== SEED {seed} =====')
    for cmd in cmds:
        print('>>>', ' '.join(cmd))
        subprocess.run(cmd, check=True)

print('\nMulti-seed runs finished ✅')

## 8) Cross-Seed Aggregation and Sample Efficiency Analysis

This final section aggregates multi-seed trajectories into summary statistics (`mean ± std`) and estimates sample efficiency in environment steps to reach a target success threshold.

The resulting tables and plots are suitable for inclusion in an academic report: they expose both central tendency and variability, which is essential for fair method comparison.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path('artifacts/multiseed')
OUT_DIR  = Path('artifacts/multiseed_aggregate')
OUT_DIR.mkdir(parents=True, exist_ok=True)

SUCCESS_THRESHOLD = 0.80
MAX_STEPS = 100
RUN_PROFILE = 'mvp'                       # должен совпадать с ячейкой выше
epu, gs = (4, 4) if RUN_PROFILE == 'mvp' else (8, 8)

# ---------- helpers ----------

def load_histories(method_subdir):
    """Загружает history.csv для каждого seed."""
    out = {}
    for sd in sorted(BASE_DIR.glob('seed_*')):
        path = sd / method_subdir / 'history.csv'
        if path.exists():
            out[int(sd.name.split('_')[-1])] = pd.read_csv(path)
    return out

def final_metrics(histories):
    rows = [{'seed': s, 'success_rate': float(df['success_rate'].iloc[-1]),
             'avg_return': float(df['avg_return'].iloc[-1])}
            for s, df in histories.items()]
    return pd.DataFrame(rows)

def mean_std_curve(histories, xcol):
    all_x = sorted(set(np.concatenate([df[xcol].values for df in histories.values()])))
    curves = []
    for df in histories.values():
        s = pd.Series(df['success_rate'].values, index=df[xcol].values)
        curves.append(s.reindex(all_x).interpolate().ffill().bfill().values)
    arr = np.array(curves)
    return np.array(all_x), arr.mean(0), arr.std(0)

def first_reach(df, threshold):
    xcol = 'update' if 'update' in df.columns else 'epoch' if 'epoch' in df.columns else None
    if xcol is None: return np.nan
    hit = df[df['success_rate'] >= threshold]
    return float(hit.iloc[0][xcol]) if len(hit) else np.nan

# ---------- load ----------

sft_h = load_histories('sft')
ga_h  = load_histories('grpo_action')
gt_h  = load_histories('grpo_text_action')

if not (sft_h and ga_h and gt_h):
    raise RuntimeError('Не найдены все history.csv. Сначала выполните секцию 7.')

# ---------- final metrics ----------

rows = []
for name, h in [('SFT', sft_h), ('GRPO-action', ga_h), ('GRPO-text+action', gt_h)]:
    fm = final_metrics(h)
    rows.append({
        'method': name, 'n_seeds': len(fm),
        'success_mean': f"{fm['success_rate'].mean():.3f}",
        'success_std':  f"{fm['success_rate'].std(ddof=1):.3f}" if len(fm) > 1 else '—',
        'return_mean':  f"{fm['avg_return'].mean():.3f}",
        'return_std':   f"{fm['avg_return'].std(ddof=1):.3f}" if len(fm) > 1 else '—',
    })
summary = pd.DataFrame(rows)

# ---------- sample efficiency ----------

se_rows = []
for name, h in [('GRPO-action', ga_h), ('GRPO-text+action', gt_h)]:
    for seed, df in h.items():
        u = first_reach(df, SUCCESS_THRESHOLD)
        eps = u * epu * gs if not np.isnan(u) else np.nan
        se_rows.append({'method': name, 'seed': seed,
                        'updates': u, 'episodes': eps,
                        'env_steps': eps * MAX_STEPS if not np.isnan(eps) else np.nan})
se_all = pd.DataFrame(se_rows)
se_summary = se_all.groupby('method').agg(
    seeds=('seed', 'count'),
    reached=('updates', lambda x: int(np.sum(~np.isnan(x)))),
    updates_mean=('updates', 'mean'),
    episodes_mean=('episodes', 'mean'),
).reset_index()

# ---------- plot ----------

x_ga, m_ga, s_ga = mean_std_curve(ga_h, 'update')
x_gt, m_gt, s_gt = mean_std_curve(gt_h, 'update')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_ga, m_ga, label='GRPO-action')
ax.fill_between(x_ga, m_ga - s_ga, m_ga + s_ga, alpha=0.2)
ax.plot(x_gt, m_gt, label='GRPO-text+action')
ax.fill_between(x_gt, m_gt - s_gt, m_gt + s_gt, alpha=0.2)
ax.axhline(SUCCESS_THRESHOLD, ls='--', lw=1, label=f'threshold={SUCCESS_THRESHOLD}')
ax.set(xlabel='Update', ylabel='Success rate')
ax.legend(); fig.tight_layout()
fig.savefig(OUT_DIR / 'grpo_success_mean_std.png', dpi=180)

# ---------- save ----------

summary.to_csv(OUT_DIR / 'final_metrics.csv', index=False)
se_all.to_csv(OUT_DIR / 'sample_efficiency_by_seed.csv', index=False)
se_summary.to_csv(OUT_DIR / 'sample_efficiency_summary.csv', index=False)

print('Saved to:', OUT_DIR)
print('\nFinal metrics (mean±std):')
display(summary)
print('\nSample efficiency:')
display(se_summary)